# Tensorflow playground with code

Remember when we played witht the tensorflow playground? Well let's do a quick exercise in order to recreate these examples but with code!

## Circle Problem

* 1️⃣ import `make_circles` from sklearn and create an object data containing circle data of 1000 observations, with some noise and a factor of your choice.



In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Generate circular data
x, y = make_circles(n_samples=1000, noise=0.1, factor=0.5)

# Convert to PyTorch tensors
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

print("data",x)

print("target",y)

data tensor([[-0.2764, -0.2580],
        [-0.0429, -0.3055],
        [-0.2058, -0.4546],
        ...,
        [-0.7724,  0.4526],
        [ 0.4504, -0.3014],
        [-0.2771, -0.0641]])
target tensor([1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0,
        0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1,
        1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1,
        1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0,
        1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
        1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0,
        1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1,
        0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1,
        0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0,
        1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
      

* 2️⃣ Visualize the data

In [29]:
from plotly import graph_objects as go
color_chart = ["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]

fig = go.Figure(data=[
    go.Scatter(
        x=x[:, 0],
        y=x[:, 1],
        mode="markers",
        marker=dict(
            color=y.numpy(),
            colorscale=color_chart[0:2]
        ),
    )
])
fig.show()

* 3️⃣ Split them in train and validation set with sklearn
* 4️⃣ Form two batch datasets, one for training data, one for validation data

In [30]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Split into training and validation sets
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

# Create DataLoader for batches
batch_size = 8
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

* 5️⃣ Create a neural network model in order to make predictions on this dataset, try and make it as simple as possible.

In [31]:
import torch.nn as nn
import torch.optim as optim

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 4)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(4, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

model = SimpleNN()
print(model)

SimpleNN(
  (fc1): Linear(in_features=2, out_features=4, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=4, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [32]:
from torchviz import make_dot

# Create a dummy input tensor
dummy_input = torch.randn(10, 2)

# Get the model output
output = model(dummy_input)

# Create and display the computational graph
make_dot(output, params=dict(model.named_parameters())).render("model_architecture", format="png")


'model_architecture.png'

* 6️⃣ Compile the model using Adam and a loss function that suits our problem.

In [33]:
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.01)

* 7️⃣ Train the model over 100 epochs

In [34]:
import torch

def train(model, train_loader, val_loader, criterion, optimizer, epochs=100):
    """
    Function to train a PyTorch model with training and validation datasets.
    
    Parameters:
    model: The neural network model to train.
    train_loader: DataLoader for the training dataset.
    val_loader: DataLoader for the validation dataset.
    criterion: Loss function (e.g., Binary Cross Entropy for classification).
    optimizer: Optimization algorithm (e.g., Adam, SGD).
    epochs: Number of training epochs (default=100).
    
    Returns:
    history: Dictionary containing loss and accuracy for both training and validation.
    """
    
    # Dictionary to store training & validation loss and accuracy over epochs
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    
    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, correct = 0, 0  # Initialize total loss and correct predictions
        
        # Training loop
        for inputs, labels in train_loader:
            optimizer.zero_grad()  # Reset gradients before each batch
            outputs = model(inputs).squeeze()  # Forward pass
            loss = criterion(outputs, labels.float())  # Compute loss
            loss.backward()  # Backpropagation (compute gradients)
            optimizer.step()  # Update model parameters
            
            total_loss += loss.item()  # Accumulate batch loss
            correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions
        
        # Compute average loss and accuracy for training
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)
        
        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0
        with torch.no_grad():  # No need to compute gradients during validation
            for inputs, labels in val_loader:
                outputs = model(inputs).squeeze()  # Forward pass
                loss = criterion(outputs, labels.float())  # Compute loss
                val_loss += loss.item()  # Accumulate validation loss
                val_correct += ((outputs > 0.5) == labels).sum().item()  # Count correct predictions
        
        # Compute average loss and accuracy for validation
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)
        
        # Store metrics in history dictionary
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        
        # Print training progress
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    return history  # Return training history

# Train the model
history = train(model, train_loader, val_loader, criterion, optimizer, epochs=100)


Epoch [1/100], Loss: 0.6793, Acc: 0.4925, Val Loss: 0.6680, Val Acc: 0.5650
Epoch [2/100], Loss: 0.6405, Acc: 0.6025, Val Loss: 0.6290, Val Acc: 0.5400
Epoch [3/100], Loss: 0.5785, Acc: 0.7025, Val Loss: 0.5375, Val Acc: 0.7800
Epoch [4/100], Loss: 0.4891, Acc: 0.7762, Val Loss: 0.4428, Val Acc: 0.8050
Epoch [5/100], Loss: 0.4109, Acc: 0.8013, Val Loss: 0.3743, Val Acc: 0.8350
Epoch [6/100], Loss: 0.3500, Acc: 0.8838, Val Loss: 0.3274, Val Acc: 0.9050
Epoch [7/100], Loss: 0.3051, Acc: 0.9137, Val Loss: 0.2908, Val Acc: 0.9300
Epoch [8/100], Loss: 0.2687, Acc: 0.9387, Val Loss: 0.2666, Val Acc: 0.9400
Epoch [9/100], Loss: 0.2409, Acc: 0.9500, Val Loss: 0.2399, Val Acc: 0.9400
Epoch [10/100], Loss: 0.2190, Acc: 0.9537, Val Loss: 0.2106, Val Acc: 0.9650
Epoch [11/100], Loss: 0.1979, Acc: 0.9625, Val Loss: 0.1946, Val Acc: 0.9650
Epoch [12/100], Loss: 0.1817, Acc: 0.9688, Val Loss: 0.1795, Val Acc: 0.9750
Epoch [13/100], Loss: 0.1707, Acc: 0.9663, Val Loss: 0.1787, Val Acc: 0.9750
Epoch [1

* 8️⃣ Plot the evolution of the train loss and the validation loss and the evolution of the train metric and the validation metric.

In [35]:
from plotly import graph_objects as go
fig = go.Figure(data=[
    go.Scatter(y=history['loss'], name="Training Loss", mode="lines", marker=dict(color=color_chart[0])),
    go.Scatter(y=history['val_loss'], name="Validation Loss", mode="lines", marker=dict(color=color_chart[1]))
])
fig.update_layout(title='Training and Validation Loss', xaxis_title='Epochs', yaxis_title='Loss')
fig.show()

In [36]:
fig = go.Figure(data=[
    go.Scatter(y=history['accuracy'], name="Training Accuracy", mode="lines", marker=dict(color=color_chart[4])),
    go.Scatter(y=history['val_accuracy'], name="Validation Accuracy", mode="lines", marker=dict(color=color_chart[5]))
])
fig.update_layout(title='Training and Validation Accuracy', xaxis_title='Epochs', yaxis_title='Accuracy')
fig.show()

* 9️⃣ Did the model overfit ?

No it seemed rather that the model underfit! We should increase the learning rate or the number of neurons on our layers.

* 🔟 Can you try and plot the decision function of the model in the data space? Like we saw in the tensorflow playground? You can get inspiration <a href="https://plotly.com/python/knn-classification/"> here </a>

In [37]:
import numpy as np
x1_range = np.linspace(-1.5, 1.5, 100)
x2_range = np.linspace(-1.5, 1.5, 100)
x1, x2 = np.meshgrid(x1_range, x2_range)
x_test = np.array([[xx1, xx2] for xx1, xx2 in zip(x1.reshape(-1), x2.reshape(-1))])

x_test_tensor = torch.tensor(x_test, dtype=torch.float32)
pred_test = model(x_test_tensor).detach().numpy().reshape(100, 100)

fig = go.Figure(data=[
    go.Contour(
        x=x1_range,
        y=x2_range,
        z=pred_test,
        colorscale=[color_chart[-2], color_chart[-1]]
    )
])
fig.add_trace(go.Scatter(x=x[:, 0], y=x[:, 1], mode="markers", marker=dict(color=y.numpy(), colorscale=color_chart[0:2])))
fig.show()

## Spiral problem

* 1️⃣ Use the following code to produce some spiral data:

In [38]:
import numpy as np
import torch
import matplotlib.pyplot as plt

N = 1000
theta = np.sqrt(np.random.rand(N)) * 4 * np.pi

r_a = 2 * theta + np.pi
data_a = np.array([np.cos(theta) * r_a, np.sin(theta) * r_a]).T
x_a = data_a + np.random.randn(N, 2)
x_a = x_a / np.abs(x_a).max()

r_b = -2 * theta - np.pi
data_b = np.array([np.cos(theta) * r_b, np.sin(theta) * r_b]).T
x_b = data_b + np.random.randn(N, 2)
x_b = x_b / np.abs(x_b).max()

res_a = np.append(x_a, np.zeros((N, 1)), axis=1)
res_b = np.append(x_b, np.ones((N, 1)), axis=1)

res = np.append(res_a, res_b, axis=0)
np.random.shuffle(res)

np.savetxt("result.csv", res, delimiter=",", header="x,y,label", comments="", fmt='%.5f')

data = torch.tensor(res[:, 0:2], dtype=torch.float32)
target = torch.tensor(res[:, -1], dtype=torch.long)

* 2️⃣ Split into train and validation set

In [39]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(data, target, test_size=0.2)

* 3️⃣ Form a train and validation batch dataset

In [40]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

In [41]:
next(iter(train_loader))

[tensor([[ 0.7204,  0.1774],
         [ 0.0694,  0.4159],
         [ 0.9531, -0.0997],
         [-0.3049,  0.3177],
         [-1.0000,  0.0068],
         [ 0.2534, -0.3642],
         [-0.3365, -0.0171],
         [-0.0231,  0.8033]]),
 tensor([1, 1, 0, 1, 1, 0, 0, 1])]

* 4️⃣ Create a neural network model that can acheive good predictions on the train set (for now we do not care about overfitting)

In [42]:
import torch.nn as nn
import torch.optim as optim

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

model = SimpleNN()

* 5️⃣ Compile the model with the right loss and metric and Adam optimizer

In [43]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

* 6️⃣ Train the model over 100 epochs

In [44]:
def train(model, train_loader, val_loader, criterion, optimizer, epochs=100):
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += ((outputs > 0.5) == labels).sum().item()
        
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss, val_correct = 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                outputs = model(inputs).squeeze()
                loss = criterion(outputs, labels.float())
                val_loss += loss.item()
                val_correct += ((outputs > 0.5) == labels).sum().item()
        
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)
        
        # Save history
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    return history

# Train the model
history = train(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch [1/100], Loss: 0.6400, Acc: 0.6181, Val Loss: 0.5744, Val Acc: 0.6475
Epoch [2/100], Loss: 0.5676, Acc: 0.6637, Val Loss: 0.5065, Val Acc: 0.7075
Epoch [3/100], Loss: 0.5049, Acc: 0.6887, Val Loss: 0.4614, Val Acc: 0.6575
Epoch [4/100], Loss: 0.3914, Acc: 0.7769, Val Loss: 0.3564, Val Acc: 0.7750
Epoch [5/100], Loss: 0.3481, Acc: 0.8194, Val Loss: 0.3574, Val Acc: 0.7950
Epoch [6/100], Loss: 0.3361, Acc: 0.8231, Val Loss: 0.3251, Val Acc: 0.8225
Epoch [7/100], Loss: 0.3092, Acc: 0.8287, Val Loss: 0.2449, Val Acc: 0.8300
Epoch [8/100], Loss: 0.2540, Acc: 0.8506, Val Loss: 0.1984, Val Acc: 0.8725
Epoch [9/100], Loss: 0.1834, Acc: 0.9206, Val Loss: 0.2363, Val Acc: 0.8975
Epoch [10/100], Loss: 0.2266, Acc: 0.9119, Val Loss: 0.2214, Val Acc: 0.9125
Epoch [11/100], Loss: 0.1707, Acc: 0.9287, Val Loss: 0.1504, Val Acc: 0.9425
Epoch [12/100], Loss: 0.1947, Acc: 0.9206, Val Loss: 0.1502, Val Acc: 0.9325
Epoch [13/100], Loss: 0.1603, Acc: 0.9350, Val Loss: 0.1406, Val Acc: 0.9350
Epoch [1

* 7️⃣ Is the model overfitting? Use visualization

In [45]:
from plotly import graph_objects as go
fig = go.Figure(data=[
    go.Scatter(y=history['loss'], name="Training Loss", mode="lines", marker=dict(color=color_chart[0])),
    go.Scatter(y=history['val_loss'], name="Validation Loss", mode="lines", marker=dict(color=color_chart[1]))
])
fig.update_layout(title='Training and Validation Loss', xaxis_title='Epochs', yaxis_title='Loss')
fig.show()

* 8️⃣ Visualize the decision boundary, would you say the model could benefit from some regularization?

In [46]:
import numpy as np
import torch
x1_example = np.linspace(-1,1,100)
x2_example = np.linspace(-1,1,100)

x1, x2 = np.meshgrid(x1_example, x2_example)
example = torch.tensor(np.c_[x1.ravel(), x2.ravel()], dtype=torch.float32)

pred_example = model(example).detach().numpy().reshape(100, 100)

fig = go.Figure(data=[
    go.Contour(
        x=x1_example,
        y=x2_example,
        z=pred_example,
        colorscale=[color_chart[-2], color_chart[-1]]
    )
])
fig.add_trace(go.Scatter(x=data[:, 0],
                         y=data[:, 1],
                         mode="markers",
                         marker=dict(
                             color=target.numpy(),
                             colorscale=color_chart[0:2])
                         )
)
fig.show()

* 9️⃣ Create a new model and add regularization to each layer, train it and visualize the decision boundary

In [47]:
class RegularizedNN(nn.Module):
    def __init__(self):
        super(RegularizedNN, self).__init__()
        self.fc1 = nn.Linear(2, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.dropout = nn.Dropout(0.01)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.sigmoid(self.fc3(x))
        return x

model = RegularizedNN()
optimizer = optim.Adam(model.parameters(), lr=0.01,weight_decay=0.001)
train(model, train_loader, val_loader, criterion, optimizer, epochs=500)

Epoch [1/500], Loss: 0.6611, Acc: 0.6050, Val Loss: 0.6287, Val Acc: 0.6275
Epoch [2/500], Loss: 0.6450, Acc: 0.6106, Val Loss: 0.6101, Val Acc: 0.6450
Epoch [3/500], Loss: 0.6194, Acc: 0.6125, Val Loss: 0.5723, Val Acc: 0.6825
Epoch [4/500], Loss: 0.5587, Acc: 0.6281, Val Loss: 0.5325, Val Acc: 0.6775
Epoch [5/500], Loss: 0.5041, Acc: 0.6725, Val Loss: 0.4196, Val Acc: 0.7500
Epoch [6/500], Loss: 0.4484, Acc: 0.7225, Val Loss: 0.4034, Val Acc: 0.7625
Epoch [7/500], Loss: 0.4196, Acc: 0.7675, Val Loss: 0.3340, Val Acc: 0.8400
Epoch [8/500], Loss: 0.3470, Acc: 0.8263, Val Loss: 0.3281, Val Acc: 0.8325
Epoch [9/500], Loss: 0.3105, Acc: 0.8650, Val Loss: 0.2680, Val Acc: 0.8900
Epoch [10/500], Loss: 0.2624, Acc: 0.8988, Val Loss: 0.2277, Val Acc: 0.9125
Epoch [11/500], Loss: 0.2392, Acc: 0.9012, Val Loss: 0.2521, Val Acc: 0.8775
Epoch [12/500], Loss: 0.2324, Acc: 0.9119, Val Loss: 0.2075, Val Acc: 0.9200
Epoch [13/500], Loss: 0.2164, Acc: 0.9225, Val Loss: 0.2163, Val Acc: 0.9175
Epoch [1

{'loss': [0.6611089609563351,
  0.6449515874683857,
  0.6193952578306198,
  0.5586747108399868,
  0.504096127897501,
  0.44838629104197025,
  0.41955714132636784,
  0.34701038068160417,
  0.31049142498522997,
  0.2623902839049697,
  0.23922189263626933,
  0.23240858119912444,
  0.2164454973861575,
  0.21621820939704775,
  0.1940915825404227,
  0.21194273460656404,
  0.18520947099663318,
  0.19285378385335206,
  0.1786329946666956,
  0.1774583972012624,
  0.17484123896807433,
  0.15980674350634216,
  0.1644640731997788,
  0.16126288912724704,
  0.16271521710790693,
  0.1399335350259207,
  0.14236790843307973,
  0.14831231868360192,
  0.14424290165305137,
  0.15692066540475935,
  0.14047322071157395,
  0.1551587177393958,
  0.15530042632017285,
  0.14419947255169974,
  0.1355835464084521,
  0.13184606935828924,
  0.13616589253302663,
  0.13814058705698698,
  0.16065876834094525,
  0.14034561739768833,
  0.1422103727515787,
  0.12681474754121155,
  0.14059346018824725,
  0.143370467964559

In [48]:
import numpy as np
import torch
x1_example = np.linspace(-1,1,100)
x2_example = np.linspace(-1,1,100)

x1, x2 = np.meshgrid(x1_example, x2_example)
example = torch.tensor(np.c_[x1.ravel(), x2.ravel()], dtype=torch.float32)

pred_example = model(example).detach().numpy().reshape(100, 100)

fig = go.Figure(data=[
    go.Contour(
        x=x1_example,
        y=x2_example,
        z=pred_example,
        colorscale=[color_chart[-2], color_chart[-1]]
    )
])
fig.add_trace(go.Scatter(x=data[:, 0],
                         y=data[:, 1],
                         mode="markers",
                         marker=dict(
                             color=target.numpy(),
                             colorscale=color_chart[0:2])
                         )
)
fig.show()